Import Libaries

In [1]:
import pandas as pd
import numpy as np
from faker import Faker
import random
from datetime import datetime, timedelta

print("All libraries imported successfully!")

All libraries imported successfully!


In [2]:
import mysql.connector

print(mysql.connector.__version__)

9.3.0


Database Connection

In [3]:
connection = mysql.connector.connect(
    host="127.0.0.1",
    port=3306,
    user="root",
    password="Ansu@0612",
    database="revenuepilot"
)

cursor = connection.cursor()

print("✅ Connected to RevenuePilot database")

✅ Connected to RevenuePilot database


Initializing faker

In [4]:
fake = Faker()

random.seed(42)
Faker.seed(42)

Data Generator - (Company table)

In [5]:
import random
from faker import Faker
import pandas as pd
from pathlib import Path

# -----------------------------
# Configuration
# -----------------------------
fake = Faker()
random.seed(42)
Faker.seed(42)

NUM_COMPANIES = 500

# -----------------------------
# Master Data
# -----------------------------
company_sizes = [
    "Startup",
    "SMB",
    "Mid-Market",
    "Enterprise"
]

company_size_weights = [40, 30, 20, 10]

industries = [
    "SaaS",
    "FinTech",
    "Healthcare",
    "EdTech",
    "E-commerce",
    "AI",
    "Cybersecurity",
    "Marketing",
    "Retail",
    "Logistics"
]

countries = [
    "United States",
    "India",
    "United Kingdom",
    "Germany",
    "Canada",
    "Australia",
    "Singapore",
    "France",
    "Japan",
    "Netherlands"
]

# -----------------------------
# Generate Companies
# -----------------------------
companies = []

for _ in range(NUM_COMPANIES):

    company_name = fake.unique.company()

    website = (
        "https://"
        + company_name.lower()
        .replace(",", "")
        .replace(".", "")
        .replace("&", "and")
        .replace(" ", "")
        + ".com"
    )

    companies.append({
        "company_name": company_name,
        "company_size": random.choices(
            company_sizes,
            weights=company_size_weights,
            k=1
        )[0],
        "country": random.choice(countries),
        "industry": random.choice(industries),
        "website": website
    })

# -----------------------------
# Save CSV
# -----------------------------
df = pd.DataFrame(companies)

output_path = Path("data/raw")
output_path.mkdir(parents=True, exist_ok=True)

df.to_csv(
    output_path / "company_raw.csv",
    index=False
)

print(f"Generated {len(df)} companies.")
print(df.head())

Generated 500 companies.
                      company_name company_size         country    industry  \
0  Rodriguez, Figueroa and Sanchez          SMB   United States  E-commerce   
1                        Doyle Ltd      Startup  United Kingdom     FinTech   
2    Mcclain, Miller and Henderson          SMB           Japan     FinTech   
3                   Davis and Sons          SMB   United States        SaaS   
4      Guzman, Hoffman and Baldwin      Startup         Germany      Retail   

                                   website  
0  https://rodriguezfigueroaandsanchez.com  
1                     https://doyleltd.com  
2    https://mcclainmillerandhenderson.com  
3                 https://davisandsons.com  
4      https://guzmanhoffmanandbaldwin.com  


In [6]:
print(df.head(75))

                       company_name company_size         country    industry  \
0   Rodriguez, Figueroa and Sanchez          SMB   United States  E-commerce   
1                         Doyle Ltd      Startup  United Kingdom     FinTech   
2     Mcclain, Miller and Henderson          SMB           Japan     FinTech   
3                    Davis and Sons          SMB   United States        SaaS   
4       Guzman, Hoffman and Baldwin      Startup         Germany      Retail   
..                              ...          ...             ...         ...   
70                       Gray Group      Startup       Singapore   Marketing   
71                     House-Glover   Mid-Market   United States     FinTech   
72                Henderson-Bernard      Startup       Australia     FinTech   
73          Stanley, Tucker and Lee      Startup         Germany      Retail   
74                        Novak PLC          SMB       Singapore  Healthcare   

                                    web

Noise factor for data (messy)

In [7]:
import random
import pandas as pd


# -----------------------------------
# Company Name Noise
# -----------------------------------

def company_name_noise(name):

    variations = [
        name.lower(),
        name.upper(),
        name.title(),
        f"{name} Inc.",
        f"{name} LLC",
        f" {name}",
        f"{name} "
    ]

    return random.choice(variations)


# -----------------------------------
# Country Noise
# -----------------------------------

def country_noise(country):

    mapping = {
        "United States": [
            "USA",
            "US",
            "U.S.A",
            " united states "
        ],

        "United Kingdom": [
            "UK",
            "U.K.",
            "Britain"
        ],

        "India": [
            "INDIA",
            " india ",
            "Bharat"
        ]
    }

    if country in mapping:
        return random.choice(mapping[country])

    return country


# -----------------------------------
# Company Size Noise
# -----------------------------------

def company_size_noise(size):

    return random.choice([
        size.lower(),
        size.upper(),
        size.title()
    ])


# -----------------------------------
# Website Noise
# -----------------------------------

def website_noise(url):

    options = [

        url.replace("https://", ""),

        url.replace("https://", "http://"),

        url.replace("https://", "www."),

        url.replace(".", ","),

        ""
    ]

    return random.choice(options)

In [8]:
def introduce_noise(df):

    df = df.copy()

    total_rows = len(df)

    # Select 15% of rows randomly
    noisy_rows = random.sample(
        range(total_rows),
        int(total_rows * 0.15)
    )

    for idx in noisy_rows:

        # Randomly choose which column to make messy
        choice = random.choice([
            "company",
            "country",
            "size",
            "website"
        ])

        if choice == "company":
            df.at[idx, "company_name"] = company_name_noise(
                df.at[idx, "company_name"]
            )

        elif choice == "country":
            df.at[idx, "country"] = country_noise(
                df.at[idx, "country"]
            )

        elif choice == "size":
            df.at[idx, "company_size"] = company_size_noise(
                df.at[idx, "company_size"]
            )

        else:
            df.at[idx, "website"] = website_noise(
                df.at[idx, "website"]
            )

    return df

In [9]:
df = introduce_noise(df)

print(df.head(10))

                      company_name company_size         country  \
0  Rodriguez, Figueroa and Sanchez          SMB   United States   
1                        Doyle Ltd      Startup  United Kingdom   
2    Mcclain, Miller and Henderson          SMB           Japan   
3                   Davis and Sons          SMB   United States   
4      Guzman, Hoffman and Baldwin      Startup         Germany   
5   Gardner, Robinson and Lawrence          SMB           Japan   
6                   Blake and Sons   Mid-Market           Japan   
7     Henderson, Ramirez and Lewis      Startup     Netherlands   
8                     Garcia-James   Mid-Market   United States   
9                    Abbott-Munoz           SMB       Australia   

        industry                                  website  
0     E-commerce  https://rodriguezfigueroaandsanchez.com  
1        FinTech                     https://doyleltd.com  
2        FinTech    https://mcclainmillerandhenderson.com  
3           SaaS      

In [10]:
from pathlib import Path

output_path = Path("data/raw")
output_path.mkdir(parents=True, exist_ok=True)

df.to_csv(
    output_path / "company_raw.csv",
    index=False
)

print("✅ company_raw.csv generated successfully!")

✅ company_raw.csv generated successfully!


In [11]:
df = introduce_noise(df)

print(df.head(10))

                      company_name company_size         country  \
0  Rodriguez, Figueroa and Sanchez          SMB   United States   
1                        Doyle Ltd      Startup  United Kingdom   
2    Mcclain, Miller and Henderson          SMB           Japan   
3                   Davis and Sons          SMB   United States   
4      Guzman, Hoffman and Baldwin      Startup         Germany   
5   Gardner, Robinson and Lawrence          SMB           Japan   
6                   Blake and Sons   Mid-Market           Japan   
7     Henderson, Ramirez and Lewis      Startup     Netherlands   
8                     Garcia-James   Mid-Market   United States   
9                    Abbott-Munoz           SMB       Australia   

        industry                                  website  
0     E-commerce  https://rodriguezfigueroaandsanchez.com  
1        FinTech                     https://doyleltd.com  
2        FinTech    https://mcclainmillerandhenderson.com  
3           SaaS      

In [12]:
print(f"Total Records: {len(df)}")

print("\nSample Data:")
print(df.sample(10))

Total Records: 500

Sample Data:
                    company_name company_size         country       industry  \
138            Chambers and Sons   Mid-Market       Australia  Cybersecurity   
63   West, Henderson and Ramirez   mid-market   United States         EdTech   
64                    Baxter Inc      Startup   United States             AI   
339    Winters, Newman and Brown          SMB         Germany      Marketing   
468               Lowery-Kennedy      Startup  United Kingdom        FinTech   
402                   Atkins PLC      Startup           India      Marketing   
149                 Salazar-Ball          SMB       Singapore         Retail   
340               Lopez-Martinez   Enterprise  United Kingdom         EdTech   
451               Bishop-Goodwin      Startup     Netherlands           SaaS   
51                    Walker LLC      Startup     Netherlands        FinTech   

                                 website  
138          https://chambersandsons.com  


In [13]:
print(df["company_size"].value_counts())

company_size
Startup       192
SMB           149
Mid-Market     87
Enterprise     44
startup         7
Smb             5
mid-market      4
ENTERPRISE      4
smb             3
MID-MARKET      2
enterprise      2
STARTUP         1
Name: count, dtype: int64


In [14]:
print(df["country"].value_counts())

country
France            58
Singapore         57
Canada            55
Australia         54
Netherlands       53
Japan             50
Germany           50
India             43
United Kingdom    38
United States     37
Bharat             2
UK                 1
INDIA              1
 india             1
Name: count, dtype: int64


In [15]:
df[df["website"] == ""]

,company_name,company_size,country,industry,website
51,Walker LLC,Startup,Netherlands,FinTech,
85,"Ferrell, Jones and Lewis",Enterprise,Netherlands,Retail,
145,Robinson-Brock,SMB,Singapore,AI,
146,"Holmes, Williams and Wright",SMB,Canada,Retail,
151,Manning Group Inc.,Startup,Netherlands,Logistics,
156,"Miller, Hernandez and Reyes",Startup,Germany,E-commerce,
164,Medina-Navarro,Startup,Japan,Marketing,
223,Hernandez PLC,Startup,France,EdTech,
455,Strickland-Shaw,Mid-Market,United Kingdom,Healthcare,


In [16]:
from pathlib import Path

output_path = Path("data/raw")
output_path.mkdir(parents=True, exist_ok=True)

df.to_csv(
    output_path / "company_raw.csv",
    index=False
)

print("✅ company_raw.csv saved successfully!")

✅ company_raw.csv saved successfully!


In [17]:
raw_file = Path("data/raw/company_raw.csv")

In [18]:
if raw_file.exists():
    print("✅ Raw file found!")
else:
    print("❌ Raw file not found.")

✅ Raw file found!


In [19]:
df = pd.read_csv(raw_file)

print("Data loaded successfully!")

Data loaded successfully!


In [20]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   company_name  500 non-null    object
 1   company_size  500 non-null    object
 2   country       500 non-null    object
 3   industry      500 non-null    object
 4   website       491 non-null    object
dtypes: object(5)
memory usage: 19.7+ KB
None


In [21]:
print(df.head())

                      company_name company_size         country    industry  \
0  Rodriguez, Figueroa and Sanchez          SMB   United States  E-commerce   
1                        Doyle Ltd      Startup  United Kingdom     FinTech   
2    Mcclain, Miller and Henderson          SMB           Japan     FinTech   
3                   Davis and Sons          SMB   United States        SaaS   
4      Guzman, Hoffman and Baldwin      Startup         Germany      Retail   

                                   website  
0  https://rodriguezfigueroaandsanchez.com  
1                     https://doyleltd.com  
2    https://mcclainmillerandhenderson.com  
3                 https://davisandsons.com  
4      https://guzmanhoffmanandbaldwin.com  


In [22]:
print(f"Rows    : {df.shape[0]}")
print(f"Columns : {df.shape[1]}")

Rows    : 500
Columns : 5


In [23]:
required_columns = [
    "company_name",
    "company_size",
    "country",
    "industry",
    "website"
]

missing = []

for col in required_columns:
    if col not in df.columns:
        missing.append(col)

if missing:
    print("❌ Missing Columns:", missing)
else:
    print("✅ All required columns are present.")

✅ All required columns are present.


In [24]:
print(df.isnull().sum())

company_name    0
company_size    0
country         0
industry        0
website         9
dtype: int64


In [25]:
print("=" * 40)
print("EXTRACT SUMMARY")
print("=" * 40)

print(f"Rows Loaded       : {len(df)}")
print(f"Columns Loaded    : {len(df.columns)}")
print(f"Duplicate Rows    : {df.duplicated().sum()}")

print("\nMissing Values:")
print(df.isnull().sum())

EXTRACT SUMMARY
Rows Loaded       : 500
Columns Loaded    : 5
Duplicate Rows    : 0

Missing Values:
company_name    0
company_size    0
country         0
industry        0
website         9
dtype: int64


In [27]:
## Read the raw data:
import pandas as pd
from pathlib import Path

raw_file = Path("data/raw/company_raw.csv")

df = pd.read_csv(raw_file)

print("✅ Raw data loaded successfully!")

✅ Raw data loaded successfully!


In [28]:
## create a working copy

df_clean = df.copy()

print("Working copy created!")

Working copy created!


In [29]:
# Remove leading/trailing spaces from all text columns
text_columns = df_clean.select_dtypes(include="object").columns

for col in text_columns:
    df_clean[col] = df_clean[col].str.strip()

print("✅ Extra spaces removed")

✅ Extra spaces removed


In [30]:
## Standardize Country Names

country_mapping = {
    "USA": "United States",
    "US": "United States",
    "U.S.A": "United States",
    "united states": "United States",

    "UK": "United Kingdom",
    "U.K.": "United Kingdom",
    "Britain": "United Kingdom",

    "INDIA": "India",
    "india": "India",
    "Bharat": "India"
}

df_clean["country"] = df_clean["country"].replace(country_mapping)

print("✅ Country names standardized")

✅ Country names standardized


In [31]:
## Standardize Company Size

size_mapping = {
    "startup": "Startup",
    "STARTUP": "Startup",

    "smb": "SMB",
    "SMB": "SMB",

    "mid-market": "Mid-Market",
    "MID-MARKET": "Mid-Market",

    "enterprise": "Enterprise",
    "ENTERPRISE": "Enterprise"
}

df_clean["company_size"] = df_clean["company_size"].replace(size_mapping)

print("✅ Company size standardized")

✅ Company size standardized


In [32]:
print(df_clean["country"].value_counts())

country
France            58
Singapore         57
Canada            55
Australia         54
Netherlands       53
Japan             50
Germany           50
India             47
United Kingdom    39
United States     37
Name: count, dtype: int64


In [33]:
print(df_clean["company_size"].value_counts())

company_size
Startup       200
SMB           152
Mid-Market     93
Enterprise     50
Smb             5
Name: count, dtype: int64


In [34]:
# Remove extra spaces
df_clean["company_size"] = df_clean["company_size"].str.strip()

# Convert to lowercase
df_clean["company_size"] = df_clean["company_size"].str.lower()

# Map to standard values
size_mapping = {
    "startup": "Startup",
    "smb": "SMB",
    "mid-market": "Mid-Market",
    "enterprise": "Enterprise"
}

df_clean["company_size"] = df_clean["company_size"].replace(size_mapping)

print("✅ Company size standardized")

✅ Company size standardized


In [35]:
print(df_clean["company_size"].value_counts())

company_size
Startup       200
SMB           157
Mid-Market     93
Enterprise     50
Name: count, dtype: int64


In [36]:
# Remove leading/trailing spaces
df_clean["company_name"] = df_clean["company_name"].str.strip()

# Remove common company suffixes
df_clean["company_name"] = (
    df_clean["company_name"]
    .str.replace(r"\s+Inc\.$", "", regex=True)
    .str.replace(r"\s+LLC$", "", regex=True)
)

# Convert to Title Case
df_clean["company_name"] = df_clean["company_name"].str.title()

print("✅ Company names cleaned")

✅ Company names cleaned


In [38]:
print(df_clean["company_name"].value_counts())

company_name
Rodriguez, Figueroa And Sanchez    1
Turner, Ortiz And Taylor           1
Harper, Smith And Buchanan         1
Davis-Lewis                        1
Lopez-Martinez                     1
                                  ..
Smith, Gilmore And Johnston        1
Stein-Silva                        1
Brown-Wall                         1
Russell-Daniels                    1
Acosta Inc                         1
Name: count, Length: 500, dtype: int64


In [39]:
# Remove leading/trailing spaces
df_clean["website"] = df_clean["website"].str.strip()

# Replace blank strings with missing values
df_clean["website"] = df_clean["website"].replace("", pd.NA)

# Convert http:// to https://
df_clean["website"] = df_clean["website"].str.replace(
    "http://",
    "https://",
    regex=False
)

# Add https:// if missing
df_clean["website"] = df_clean["website"].apply(
    lambda x: f"https://{x}"
    if pd.notna(x) and not x.startswith("https://")
    else x
)

print("✅ Website URLs cleaned")

✅ Website URLs cleaned


In [41]:
print(df_clean["website"].value_counts())

website
https://rodriguezfigueroaandsanchez.com    1
https://willisandsons.com                  1
https://zhangplc.com                       1
https://harpersmithandbuchanan.com         1
https://davis-lewis.com                    1
                                          ..
https://johnson-rogers.com                 1
https://hurstfreemanandnelson.com          1
https://castaneda-ashley.com               1
https://smith-bell.com                     1
https://acostainc.com                      1
Name: count, Length: 491, dtype: int64


In [42]:
df_clean["website"].sample(10)

339      https://wintersnewmanandbrown.com
27     https://hoffmanbakerandrichards.com
481      https://hartmanromeroandsmith.com
183          https://rodriguez-johnson.com
123                   https://scottltd.com
41                 https://georgegroup.com
151                                    NaN
92               https://newtonandsons.com
16                  https://jamesgroup.com
370                https://reynoldsltd.com
Name: website, dtype: object

In [43]:
print(df_clean.isnull().sum())

company_name    0
company_size    0
country         0
industry        0
website         9
dtype: int64


In [44]:
# Remove rows missing essential fields
df_clean = df_clean.dropna(
    subset=[
        "company_name",
        "company_size",
        "country",
        "industry"
    ]
)

print("✅ Missing values handled")

✅ Missing values handled


In [45]:
print(df_clean.isnull().sum())

company_name    0
company_size    0
country         0
industry        0
website         9
dtype: int64


In [ ]:
## Remove duplicates

In [46]:
print(f"Rows before removing duplicates: {len(df_clean)}")

duplicate_count = df_clean.duplicated().sum()

print(f"Duplicate rows: {duplicate_count}")

Rows before removing duplicates: 500
Duplicate rows: 0


In [47]:
df_clean = df_clean.drop_duplicates()

print("✅ Duplicate rows removed")

✅ Duplicate rows removed


In [48]:
print(f"Rows after removing duplicates: {len(df_clean)}")

Rows after removing duplicates: 500


In [49]:
print("=" * 40)
print("TRANSFORM SUMMARY")
print("=" * 40)

print(f"Rows           : {len(df_clean)}")
print(f"Columns        : {len(df_clean.columns)}")
print(f"Duplicates     : {df_clean.duplicated().sum()}")

print("\nMissing Values:")
print(df_clean.isnull().sum())

TRANSFORM SUMMARY
Rows           : 500
Columns        : 5
Duplicates     : 0

Missing Values:
company_name    0
company_size    0
country         0
industry        0
website         9
dtype: int64


In [50]:
from pathlib import Path

processed_path = Path("data/processed")
processed_path.mkdir(parents=True, exist_ok=True)

df_clean.to_csv(
    processed_path / "company_clean.csv",
    index=False
)

print("✅ company_clean.csv exported successfully!")

✅ company_clean.csv exported successfully!


In [ ]:
Phase 4 - (Load)

In [51]:
import pandas as pd

df = pd.read_csv("data/processed/company_clean.csv")

print(df.head())

                      company_name company_size         country    industry  \
0  Rodriguez, Figueroa And Sanchez          SMB   United States  E-commerce   
1                        Doyle Ltd      Startup  United Kingdom     FinTech   
2    Mcclain, Miller And Henderson          SMB           Japan     FinTech   
3                   Davis And Sons          SMB   United States        SaaS   
4      Guzman, Hoffman And Baldwin      Startup         Germany      Retail   

                                   website  
0  https://rodriguezfigueroaandsanchez.com  
1                     https://doyleltd.com  
2    https://mcclainmillerandhenderson.com  
3                 https://davisandsons.com  
4      https://guzmanhoffmanandbaldwin.com  


In [53]:
import mysql.connector

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="Ansu@0612",
    database="revenuepilot"
)

cursor = conn.cursor()

print("✅ Connected to MySQL")

✅ Connected to MySQL


In [54]:
insert_query = """
INSERT INTO company
(
company_name,
company_size,
country,
industry,
website
)
VALUES
(%s,%s,%s,%s,%s)
"""

In [55]:
## Load DATA

for _, row in df.iterrows():

    cursor.execute(
        insert_query,
        (
            row["company_name"],
            row["company_size"],
            row["country"],
            row["industry"],
            row["website"]
        )
    )

conn.commit()

print("✅ Data Loaded Successfully")

✅ Data Loaded Successfully


In [56]:
cursor.execute("SELECT COUNT(*) FROM company")

print(cursor.fetchone())

(500,)


In [57]:
cursor.close()
conn.close()

print("✅ Connection Closed")

✅ Connection Closed
